In [1]:
# Load environment variables and verify the project setup.
import sys
from pathlib import Path

# Find the repo root (the folder containing env_checker.py) and make it importable.
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "env_checker.py").exists())
sys.path.insert(0, str(ROOT))

# Load .env into the environment for this session.
try:
    from dotenv import load_dotenv
    load_dotenv(ROOT / ".env")
except ModuleNotFoundError:
    print("python-dotenv not installed yet — run: uv add python-dotenv")

# Verify .env variables and required packages.
from env_checker import run_checks
run_checks()

Environment variables (from .env.example)
  ✓ OPENAI_API_KEY  — set
  ✓ ANTHROPIC_API_KEY  — set
  ✓ LANGSMITH_TRACING  — set
  ✓ LANGSMITH_ENDPOINT  — set
  ✓ LANGSMITH_API_KEY  — set
  ✓ LANGSMITH_PROJECT  — set
  ✓ CHROMA_PERSIST_DIR  — set

Required packages (from pyproject.toml)
  ✓ beautifulsoup4  — installed (4.14.3)
  ✓ chromadb  — installed (1.5.9)
  ✓ langchain  — installed (1.3.2)
  ✓ langchain-chroma  — installed (1.1.0)
  ✓ langchain-classic  — installed (1.0.7)
  ✓ langchain-community  — installed (0.4.2)
  ✓ langchain-core  — installed (1.4.0)
  ✓ langchain-experimental  — installed (0.4.2)
  ✓ langchain-openai  — installed (1.2.2)
  ✓ lxml  — installed (6.1.1)
  ✓ onnxruntime  — installed (1.19.2)
  ✓ pypdf  — installed (6.12.2)
  ✓ python-dotenv  — installed (1.2.2)
  ✓ rank-bm25  — installed (0.2.2)
  ✓ rapidfuzz  — installed (3.14.5)
  ✓ ipykernel  — installed (7.2.0)
  ✓ jupyterlab  — installed (4.5.7)

✓ All checks passed.


True

# Faithfulness

**Faithfulness** = the fraction of factual claims in the *answer* that are actually supported by the *retrieved context*. It catches hallucination: an answer that states things the context doesn't back up scores low.

We compute it as **LLM-as-judge** (the same approach RAGAS uses internally): split the answer into atomic claims, then verify each against the context.

> RAGAS has a built-in `Faithfulness` metric, but `ragas` currently imports a removed `langchain_community` path and won't load on this LangChain 1.x stack, so we implement the metric directly.

## Example to evaluate

An answer generated from some retrieved context — note one unsupported claim.

In [2]:
context = (
    "Low-Level Design (LLD) provides the detailed internal design of each component — "
    "the blueprint developers code from. It specifies class/module structure, methods, "
    "data structures and database fields."
)
answer = (
    "LLD describes the detailed internal design of each component, including classes, "
    "methods and database fields. It is written by the QA team."   # <- unsupported
)

## Judge: claim extraction + verification

In [3]:
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field

class ClaimVerdict(BaseModel):
    claim: str = Field(description="an atomic factual claim taken from the answer")
    supported: bool = Field(description="True if the claim can be inferred from the context")
    reason: str

class FaithfulnessReport(BaseModel):
    verdicts: list[ClaimVerdict]

judge = ChatOpenAI(model="gpt-4o-mini", temperature=0).with_structured_output(FaithfulnessReport)

PROMPT = """You evaluate the FAITHFULNESS of an answer against its retrieved context.
Step 1: split the ANSWER into atomic factual claims.
Step 2: for each claim, decide if it can be directly inferred from the CONTEXT.

CONTEXT:
{context}

ANSWER:
{answer}
"""

report = judge.invoke(PROMPT.format(context=context, answer=answer))

## Score

faithfulness = supported claims / total claims (1.0 = fully grounded).

In [4]:
for v in report.verdicts:
    print(("OK " if v.supported else "XX"), v.claim)

supported = sum(v.supported for v in report.verdicts)
faithfulness = supported / len(report.verdicts)
print(f"\nFaithfulness = {supported}/{len(report.verdicts)} = {faithfulness:.2f}")

OK  LLD describes the detailed internal design of each component.
OK  LLD includes classes, methods and database fields.
XX LLD is written by the QA team.

Faithfulness = 2/3 = 0.67
